# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Loading the dataset**

In [1]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [2]:
table = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Two signals checked first, against the Feb→March decline label:**
1. **CTR vs. position** (behind the CTR-fix flag logic) — bucket by `position_tier_feb`,
   show mean `ctr_feb` and `n` per bucket. Already CONFIRMED on the starter CSV
   (corr = -0.239); this re-tests it on real warehouse data.
2. **Volume** (behind the quick-win flag logic) — bucket `impressions_feb` into quartiles,
   show mean `is_declining` rate and `n` per bucket. Tests whether high-visibility pages are
   actually more worth prioritizing, or just noisier.

**Excluded on purpose:** staleness/days-since-update — no such field exists in
`fact_content_daily_performance`. Testing it would mean inventing data, so it's left out
until a content-dimension table with a real update timestamp is confirmed to exist.

**The rule, in plain words (working hypothesis, pending the signal-check verdicts below):**
flag any page that's genuinely visible (impressions_feb at or above the median) but ranking
outside the top 10 (avg_position_feb > 10) — real search demand the page isn't yet capturing.
Score by how much visibility is being wasted (impressions_feb), so the highest-volume
underperformers sort to the top. **One reason code:** `visible_but_low_rank`. **One action:**
`refresh_for_ranking`.

If the volume signal check below comes back OPPOSITE or FALSE, this rule needs to change —
it shouldn't survive on a signal that didn't hold up.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd

PRIOR_MONTH, RECENT_MONTH = "2026-02", "2026-03"

# Signal 1: CTR vs position tier
sig1 = con.sql(f"""
    WITH feb AS (
        SELECT content_hash_id,
               AVG(gsc_avg_position) AS avg_position_feb,
               SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions),0) AS ctr_feb
        FROM read_parquet('{table}') WHERE month = '{PRIOR_MONTH}'
        GROUP BY content_hash_id
    )
    SELECT CASE
             WHEN avg_position_feb IS NULL THEN 'no_position'
             WHEN avg_position_feb <= 3  THEN 'top_3'
             WHEN avg_position_feb <= 10 THEN 'page_1'
             WHEN avg_position_feb <= 20 THEN 'page_2'
             ELSE 'deep' END AS position_tier,
           COUNT(*) AS n,
           ROUND(AVG(ctr_feb), 4) AS mean_ctr
    FROM feb GROUP BY 1 ORDER BY mean_ctr DESC
""")
print("Signal 1 -- CTR by position tier:")
print(sig1)
print("Verdict (fill in after reading the table above): CONFIRMED / OPPOSITE / MIXED / FALSE")

# Signal 2: volume vs decline rate
panel = con.sql(f"""
    WITH feb AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb,
               SUM(gsc_clicks) AS clicks_feb
        FROM read_parquet('{table}') WHERE month = '{PRIOR_MONTH}' GROUP BY content_hash_id
    ), mar AS (
        SELECT content_hash_id, SUM(gsc_clicks) AS clicks_mar
        FROM read_parquet('{table}') WHERE month = '{RECENT_MONTH}' GROUP BY content_hash_id
    )
    SELECT feb.content_hash_id, impressions_feb, clicks_feb, clicks_mar,
           CASE WHEN clicks_mar < clicks_feb THEN 1 ELSE 0 END AS is_declining
    FROM feb JOIN mar USING (content_hash_id)
""").df()

panel["volume_quartile"] = pd.qcut(
    panel["impressions_feb"].rank(method="first"),
    4,
    labels=["q1_low", "q2", "q3", "q4_high"]
)
sig2 = panel.groupby("volume_quartile", observed=True).agg(n=("is_declining","size"), decline_rate=("is_declining","mean")).round(3)
print("\nSignal 2 -- decline rate by volume quartile:")
print(sig2)
print("Verdict (fill in after reading the table above): CONFIRMED / OPPOSITE / MIXED / FALSE")

Signal 1 -- CTR by position tier:
┌───────────────┬────────┬──────────┐
│ position_tier │   n    │ mean_ctr │
│    varchar    │ int64  │  double  │
├───────────────┼────────┼──────────┤
│ top_3         │  19243 │   0.0106 │
│ page_1        │  75898 │   0.0052 │
│ page_2        │  31699 │   0.0031 │
│ deep          │  26719 │   0.0025 │
│ no_position   │ 167987 │     NULL │
└───────────────┴────────┴──────────┘

Verdict (fill in after reading the table above): CONFIRMED / OPPOSITE / MIXED / FALSE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Signal 2 -- decline rate by volume quartile:
                     n  decline_rate
volume_quartile                     
q1_low           75893         0.000
q2               75893         0.000
q3               75893         0.047
q4_high          75893         0.280
Verdict (fill in after reading the table above): CONFIRMED / OPPOSITE / MIXED / FALSE


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Encoding the rule from Section 1 as a score, one reason code, one action.** A page is
flagged when it clears both conditions at once — `visible` (impressions_feb at or above the
February median) and `low_rank` (avg_position_feb outside the top 10). Only flagged pages get
scored; unflagged pages score 0 and sort to the bottom. The score itself is
`impressions_feb` on flagged pages — so among everything worth flagging, the pages with the
most wasted visibility rank highest. **Reason code:** `visible_but_low_rank`. **Action:**
`refresh_for_ranking` when flagged, `monitor` otherwise. The full ranked queue is written to
`work/outputs/baseline_action_score.csv`, sorted highest score first — this file is the
actual deliverable, not just the printed preview.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

feb = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position) AS avg_position_feb,
           SUM(gsc_impressions)  AS impressions_feb
    FROM read_parquet('{table}') WHERE month = '{PRIOR_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

median_impressions = feb["impressions_feb"].median()
feb["visible"] = (feb["impressions_feb"] >= median_impressions).astype(int)
feb["low_rank"] = (feb["avg_position_feb"] > 10).astype(int)
feb["flag"] = feb["visible"] * feb["low_rank"]

feb["baseline_action_score"] = feb["flag"] * feb["impressions_feb"]
feb["reason_code"] = np.where(feb["flag"] == 1, "visible_but_low_rank", "none")
feb["action"] = np.where(feb["flag"] == 1, "refresh_for_ranking", "monitor")

queue = feb.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["client_hash_id", "content_hash_id", "avg_position_feb", "impressions_feb",
            "baseline_action_score", "reason_code", "action"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Median impressions_feb (visibility threshold): {median_impressions:.0f}")
print(f"Flagged pages: {feb['flag'].sum()} of {len(feb)}")
print("Wrote work/outputs/baseline_action_score.csv")
queue[out_cols].head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Median impressions_feb (visibility threshold): 0
Flagged pages: 58418 of 321546
Wrote work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,avg_position_feb,impressions_feb,baseline_action_score,reason_code,action
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,13.725133,162129.0,162129.0,visible_but_low_rank,refresh_for_ranking
1,client_23a62021009f63c4,content_36e53e9c707674fc,34.040988,100736.0,100736.0,visible_but_low_rank,refresh_for_ranking
2,client_23a62021009f63c4,content_df47d1b976106de4,18.008694,85935.0,85935.0,visible_but_low_rank,refresh_for_ranking
3,client_fef1a8f436438636,content_84a6bf3578312e90,20.050958,79986.0,79986.0,visible_but_low_rank,refresh_for_ranking
4,client_23a62021009f63c4,content_5e1c049f62e33b11,16.643724,73072.0,73072.0,visible_but_low_rank,refresh_for_ranking
5,client_fef1a8f436438636,content_ba462518dad435fc,27.753576,69134.0,69134.0,visible_but_low_rank,refresh_for_ranking
6,client_23a62021009f63c4,content_3df3f32f3fd58dea,24.899933,68216.0,68216.0,visible_but_low_rank,refresh_for_ranking
7,client_23a62021009f63c4,content_b51957d7f4abe47e,27.816678,57558.0,57558.0,visible_but_low_rank,refresh_for_ranking
8,client_23a62021009f63c4,content_bdf60c86117079be,33.418919,51346.0,51346.0,visible_but_low_rank,refresh_for_ranking
9,client_fef1a8f436438636,content_0aaa197051f58d6f,34.694009,49903.0,49903.0,visible_but_low_rank,refresh_for_ranking


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 10 flagged pages: the action, why it's there (grounded in its actual
avg_position_feb and impressions_feb), and what specific fact would make this pick wrong —
not a generic disclaimer, but a real condition (e.g. seasonal content, a page mid-migration)
that the rule can't see.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue[queue["flag"] == 1].head(10)

for i, row in top10.iterrows():
    print(f"#{i+1}  content={row['content_hash_id'][:12]}...  action={row['action']}")
    print(f"     reason: {row['reason_code']} -- avg_position={row['avg_position_feb']:.1f} "
          f"(outside top 10), impressions={row['impressions_feb']:.0f} (>= median {median_impressions:.0f})")
    print(f"     confidence: {'higher' if row['avg_position_feb'] > 20 else 'moderate'} "
          f"-- position is {'well' if row['avg_position_feb'] > 20 else 'only just'} past page 1")
    print(f"     wrong if: this page is intentionally low-priority (seasonal, deprecated, "
          f"or already scheduled for removal) -- the rule can't see intent, only position and volume")
    print()


#1  content=content_e8a5...  action=refresh_for_ranking
     reason: visible_but_low_rank -- avg_position=13.7 (outside top 10), impressions=162129 (>= median 0)
     confidence: moderate -- position is only just past page 1
     wrong if: this page is intentionally low-priority (seasonal, deprecated, or already scheduled for removal) -- the rule can't see intent, only position and volume

#2  content=content_36e5...  action=refresh_for_ranking
     reason: visible_but_low_rank -- avg_position=34.0 (outside top 10), impressions=100736 (>= median 0)
     confidence: higher -- position is well past page 1
     wrong if: this page is intentionally low-priority (seasonal, deprecated, or already scheduled for removal) -- the rule can't see intent, only position and volume

#3  content=content_df47...  action=refresh_for_ranking
     reason: visible_but_low_rank -- avg_position=18.0 (outside top 10), impressions=85935 (>= median 0)
     confidence: moderate -- position is only just past page

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** any top-10 entry sitting right at the visibility median or just past the
position-10 cutoff is a boundary case — small measurement noise could flip it in or out of
the flagged set. Those are the ones to sanity-check by eye before trusting the queue,
not the ones deep in `deep` position with high volume, which are unambiguous.

**Leakage check:** the rule's inputs are `avg_position_feb` and `impressions_feb` — both
built from `WHERE month = PRIOR_MONTH` only. `RECENT_MONTH` and `clicks_mar` never appear
in the scoring code above; the label used to test the signals in Section 1 was never fed
into the rule itself in Section 2. No product/team flags exist in the raw schema to leak
from in the first place.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Boundary/weak picks: within 10% of either threshold
weak = top10[
    (top10["impressions_feb"] < median_impressions * 1.10) |
    (top10["avg_position_feb"] < 11)
]
print(f"Weak/boundary picks in top 10: {len(weak)}")
print(weak[["content_hash_id", "avg_position_feb", "impressions_feb"]])

# Leakage check: confirm no recent-month or label fields in the final output
leak_terms = ["mar", "recent", "clicks_mar", "is_declining"]
leaked_cols = [c for c in out_cols if any(t in c.lower() for t in leak_terms)]
print(f"\nColumns in output CSV matching leakage patterns: {leaked_cols if leaked_cols else 'None found'}")
print(f"Scoring query WHERE clause used: month = '{PRIOR_MONTH}' only")


Weak/boundary picks in top 10: 0
Empty DataFrame
Columns: [content_hash_id, avg_position_feb, impressions_feb]
Index: []

Columns in output CSV matching leakage patterns: None found
Scoring query WHERE clause used: month = '2026-02' only


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.